---
---
# **Procesamiento Automático de Balances Generales y Estados de Resultados Mediante un Modelo de Inteligencia Artificial**
---
# Prueba de Concepto Usando Microsoft Azure AI Document Intelligence y Microsoft Azure OpenAI

---
---
##Universidad Alfonso X El Sabio
##Master en Inteligencia Artificial 2025/2026
---
##Trabajo de Fin de Master (TFM)
---
**Autor:** Raúl Sánchez -
**eMail:** rsancgom@myuax.com -
**NP:** 867196

**Tutor:** Ángel Manuel Rayo Acevedo -
**eMail:** arayoace@uax.es

**Fecha de Entrega:** 22-Junio-2026

---

## Objetivo

Construir una Prueba de Concepto (POC, por sus siglas en inglés) con el uso de la tecnología Microsoft Azure AI Document Intelligence, para extraer información financiera de empresas colombianas proveniente de archivos no editables con formatos y estructuras desconocidas, para convertirla posteriormente en datos manipulables que se puedan validar y cargar a estructuras estándar previamente definidas.

----
## Justificación
Una de los principales procesos en empresas que comercializan información empresarial, consiste en recopilar información empresarial de todo tipo para su correspondiente tratamiento, depuración, limpieza y volcado sobre bases de datos puras, comunmente llamadas Bases de Datos Unificada de Empresas, con formatos y estructuras estándar previamente definidas; esta información constituye el activo principal de esta compañías y es finalmente el producto que comercializan con sus diferentes clientes.

El mayor de los inconvenientes que se tiene corresponde al cargue manual de información financiera, específicamente Balances Generales y Estados de Resultados de las diferentes empresas en Colombia, provenientes de fuentes públicas de difícil legibilidad, en archivos PDF con imágenes no editables y con formatos y estructuras desconocidas.

Esta manualidad ocasiona una alta carga operativa derivada de la necesidad de disponer de un número importante de personas realizando estas labores, con la posibilidad de cometer errores frecuentes y con altos tiempos procesamiento debido a la transcripción de cada uno de los datos de entrada y a la interpretación de estos para llevarlos a las estructuras estándar definidas en formatos predefinidos en Excel, en donde manualmente se hacen validaciones de consistencia, totalización de cifras, comparación con las fuentes y cargue final a la Base de Datos Unificada de Empresas.

Con base en lo anterior y con el fin de cubrir estas debilidades, se planteó la posibilidad de implementar una prueba de concepto que a través de un modelo de Inteligencia Artificial apoyado en la tecnología de Microsoft Azure AI Document Intelligence, permita extraer de manera automática la información en cuestión, convertirla en datos manipulables y cargarla en estructuras estándar previamente definidas.

Las tecnologías de Inteligencia Artificial disponibles en la actualidad tales como AI-OCR (Optical Character Recognition basado en IA) y LLM (Large Language Model), pueden resolver sin lugar a duda, la problemática asociada a procesos manuales como el descrito anteriormente, con altos niveles de precisión y bajos costos.

----
##Metodología
Para cumplir con el objetivo serán desarrolladas las siguientes actividades:
1) Actividades previas.
2) Análisis y extracción del archivo fuente con Microsoft Azure AI Document Intelligence.
    - Definición y consumo de los servicios de Microsoft Azure AI Document Intelligence.
    - Formateo de la información y cargue en dataset local.
    - Exportación de la información formateada a archivo Excel.
3) Interpretación y Mapeo del Contenido con Microsoft Azure OpenAI.
    - Lectura y limpieza del documento extraído.
    - Lectura de la taxonomía definida
    - Definición y configuración del LLM Azure OpenAI para interpretar los estados financieros leídos con AI-OCR según la taxonomía.
    - Mapeo Inteligente para interpretar los estados financieros leídos con AI-OCR según la taxonomía usando LLM Azure OpenAI.
    - Generación del archivo excel final completamente interpretado y mapeado.

----

----
# **Actividades Previas**

----

In [62]:
# Instalar e importar las librerías requeridas para la POC

# Instalar librerías para el reconocimiento óptico de Azure AI Documents
import sys
!{sys.executable} -m pip install azure-ai-formrecognizer
!pip install azure-ai-formrecognizer pandas openpyxl
!pip install --upgrade azure-ai-formrecognizer
!pip install azure-ai-documentintelligence

# Instalar librerías para el LLM Azure OpenAI
!pip install openai pandas openpyxl tiktoken

# Importar librerías para el reconocimiento óptico con Azure AI Documents
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.core.credentials import AzureKeyCredential

# Importar otras librerías de propósito general
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import re
import json

# Importar librerías para el LLM Azure OpenAI
import openai
from collections import defaultdict
from openai import OpenAI
from openai import AzureOpenAI


In [63]:
# Conectar con la unidad de drive de Google y establecer directorio de trabajo

print("Montando el directorio de trabajo ...\n")
from google.colab import drive
drive.mount('/content/drive')

# Asignar variable global para el directorio de trabajo
directorio_datos = '/content/drive/MyDrive/Colab Notebooks/'

print("Directorio de trabajo montado.")

Montando el directorio de trabajo ...

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio de trabajo montado.


In [64]:
# Configurar los nombres de los archivos a utilizar

# Archivo de alta complejidad, mediana legibilidad
#pdf_path = os.path.join(directorio_datos, 'PRUEBA_12.pdf')
#excel_out = os.path.join(directorio_datos, 'PRUEBA_12_Azure201.xlsx')
#output_file = os.path.join(directorio_datos, 'PRUEBA_12_LLM201.xlsx')

# Archivo de baja complejidad, buena legibilidad
#pdf_path = os.path.join(directorio_datos, 'PRUEBA_14.pdf')
#excel_out = os.path.join(directorio_datos, 'PRUEBA_14_Azure202.xlsx')
#output_file = os.path.join(directorio_datos, 'PRUEBA_14_LLM202.xlsx')

# Archivo de baja complejidad, muy mala legibilidad
#pdf_path = os.path.join(directorio_datos, 'PRUEBA_32.pdf')
#excel_out = os.path.join(directorio_datos, 'PRUEBA_32_Azure203.xlsx')
#output_file = os.path.join(directorio_datos, 'PRUEBA_32_LLM203.xlsx')

# Archivo de alta complejidad, mediana legibilidad
pdf_path = os.path.join(directorio_datos, 'PRUEBA_03.pdf')
excel_out = os.path.join(directorio_datos, 'PRUEBA_03_Azure204.xlsx')
output_file = os.path.join(directorio_datos, 'PRUEBA_03_LLM204.xlsx')

file_input = excel_out
file_taxonomy = os.path.join(directorio_datos, '03_Taxonomia-Base-BGyEERR.xlsx')

In [65]:
# Configurar los parámetros requeridos para el servicio Google AI Documents
# Esto requiere una configuración previa en Azure Cloud
endpoint = "https://RS001-DocumentsTest.cognitiveservices.azure.com/"
key = "88tgMWd2dL4JT9Kj8y7GJWJsE8AxCotw13IgclBkXBV3xXTApLFEJQQJ99CCACHYHv6XJ3w3AAALACOGP3fJ"

# configuración Azure OpenAI
# Esto requiere una configuración previa en Azure Cloud
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://rs002.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2025-01-01-preview"
os.environ["AZURE_OPENAI_API_KEY"] = "7gfqfiCbfUbAfCwbkzcM13mK43TsHqxoFDD1f32YBYtqv34OvjrhJQQJ99CEACHYHv6XJ3w3AAABACOGxmO9"
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-02-15-preview"
DEPLOYMENT_NAME = "gpt-4o"

----
# **Análisis y Extracción del Archivo Fuente con Microsoft Azure AI Document Intelligence**

----

----
#Definición y consumo de los servicios de Microsoft Azure AI Document Intelligence

----

In [66]:
# Definir consumo del servicio Azure AI Document Intelligence que permite análisis y extracción de información del archivo

# Crea el cliente
client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

# Abre el archivo PDF y analiza con el modelo prebuilt-layout
with open(pdf_path, "rb") as f:
    pdf_content = f.read()

poller = client.begin_analyze_document(
    model_id="prebuilt-layout",
    body=pdf_content,  # Contenido binario del PDF
    pages="1-99"  # La versión F0 tiene limitaciones en el # de páginas a leer
)

result = poller.result()

num_paginas = len(result.pages)
print(f"Número de páginas leídas: {num_paginas}")

Número de páginas leídas: 2


----
#Formateo de la información y cargue en dataset local

----

In [67]:
# Construir función para transformar la salida a un dataframe de pandas con la siguiente estructura
#   pagina: número de página del documento
#	  fila: número que identifica la fila en donde debe ir el texto encontrado (para un archivo excel)
#   columna: letra que identifica la columna en donde debe ir el texto encontrado (para un archivo excel)
#   valor: texto encontrado y convertido en editable
#   Coordenadas del polígono en donde se encontró el texto:
#     x1,y1	(coordenada izquierda-inferior)
#     x2,y2	(coordenada derecha-inferior)
#     x3,y3	(coordenada derecha-superior)
#     x4,y4 (coordenada izquierda-superior)

rows = []
for page in result.pages:
    if hasattr(page, "lines") and page.lines:
        for idx, line in enumerate(page.lines, start=1):
            pts = None
            if hasattr(line, "polygon") and line.polygon:
                pts = []
                for i in range(0, len(line.polygon), 2):
                    pts.append((line.polygon[i], line.polygon[i+1]))

            if pts:
                pts = list(pts[:4])
                if len(pts) < 4:
                    pts += [(None, None)] * (4 - len(pts))
            else:
                pts = [(None, None)] * 4

            (x1, y1), (x2, y2), (x3, y3), (x4, y4) = pts

            rows.append({
                "pagina": page.page_number,
                "fila": 0,
                "columna": 0,
                "texto": line.content if hasattr(line, "content") else "",
                "x1": x1, "y1": y1,
                "x2": x2, "y2": y2,
                "x3": x3, "y3": y3,
                "x4": x4, "y4": y4,
            })

df = pd.DataFrame(
    rows,
    columns=["pagina", "fila", "columna", "texto", "x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4"]
)

In [68]:
# Mostrar resultado inicial del formateo
df

,pagina,fila,columna,texto,x1,y1,x2,y2,x3,y3,x4,y4
0,1,0,0,MUNICIPIO DE LA ESTRELLA,2.9437,0.7307,4.9718,0.7346,4.9715,0.8829,2.9434,0.8786
1,1,0,0,ESTADO DE SITUACION FINANCIERA,2.6609,0.9066,5.2273,0.9116,5.2270,1.0651,2.6606,1.0600
2,1,0,0,SEPTIEMBRE 30 DE 2023,3.0685,1.0885,4.8062,1.0950,4.8057,1.2439,3.0679,1.2374
3,1,0,0,(Cifras en pesos),3.3562,1.2761,4.5234,1.2812,4.5227,1.4424,3.3555,1.4373
4,1,0,0,jun-23,4.4577,1.6108,4.8654,1.6057,4.8672,1.7485,4.4595,1.7536
...,...,...,...,...,...,...,...,...,...,...,...,...
248,2,0,0,6.287.244.633,4.6609,9.7637,5.4984,9.7675,5.4979,9.8798,4.6604,9.8760
249,2,0,0,6.292.028.387,6.2834,9.7648,7.1289,9.7677,7.1285,9.8883,6.2830,9.8853
250,2,0,0,2990 otros pasivos diferidos,1.0242,9.9006,2.6812,9.9105,2.6804,10.0527,1.0238,10.0429
251,2,0,0,43.823.426.293,4.5888,9.9201,5.5010,9.9244,5.5004,10.0451,4.5883,10.0408


In [69]:
# Convertir el dataframe a un archivo excel con la estructura del PDF leído

# Configurar la tolerancia
TOL_X = 0.37
TOL_Y = 0.05

# Función para convertir números a columnas tipo Excel
def num_to_col(n):
    result = ""
    while n > 0:
        n, r = divmod(n - 1, 26)
        result = chr(65 + r) + result
    return result

# Función para asignar correctamente las filas
def asignar_filas(page_df):
    page_df = page_df.copy()

    # Promedio Y de cada caja
    page_df["y_mean"] = page_df[["y1", "y2", "y3", "y4"]].mean(axis=1)

    # Orden ascendente: Y menor → fila más arriba → fila 1
    valores = sorted(page_df["y_mean"].unique())

    filas = []
    fila_actual = 1

    while valores:
        base = valores[0]

        # Agrupación por tolerancia
        similares = [v for v in valores if abs(v - base) <= TOL_Y]

        for v in similares:
            filas.append((v, fila_actual))

        valores = [v for v in valores if v not in similares]
        fila_actual += 1

    fila_map = dict(filas)
    page_df["fila"] = page_df["y_mean"].map(fila_map)

    return page_df.drop(columns=["y_mean"])

# Función para asignar correctamente las columnas
def asignar_columnas(page_df):
    page_df = page_df.copy()

    page_df["x_left"] = page_df[["x1", "x4"]].min(axis=1)

    valores = sorted(page_df["x_left"].unique())

    columnas = []
    col_actual = 1

    while valores:
        base = valores[0]
        similares = [v for v in valores if abs(v - base) <= TOL_X]

        for v in similares:
            columnas.append((v, col_actual))

        valores = [v for v in valores if v not in similares]
        col_actual += 1

    col_map = dict(columnas)
    page_df["col_num"] = page_df["x_left"].map(col_map)
    page_df["columna"] = page_df["col_num"].apply(num_to_col)

    return page_df.drop(columns=["x_left", "col_num"])

# Función para procesar el formateo
def procesar(df):
    df = df.copy().reset_index(drop=True)
    df["orden_original"] = np.arange(len(df))  # Para preservar el orden

    partes = []

    for pagina, page_df in df.groupby("pagina", sort=False):
        page_df = page_df.copy()

        # Calcular filas y columnas (método usando las toleracias de filas y columnas)
        page_df = asignar_filas(page_df)
        page_df = asignar_columnas(page_df)

        partes.append(page_df)

    df_1 = pd.concat(partes, ignore_index=True)

    # Regresar el orden exacto del df original
    df_1 = df_1.sort_values("orden_original").drop(columns=["orden_original"])

    return df_1

# Generar df_1 desde df
df_1 = procesar(df)

In [70]:
# Mostrar el resultado final del formateo

df_1.head(50)

,pagina,fila,columna,texto,x1,y1,x2,y2,x3,y3,x4,y4
0,1,1,B,MUNICIPIO DE LA ESTRELLA,2.9437,0.7307,4.9718,0.7346,4.9715,0.8829,2.9434,0.8786
1,1,2,B,ESTADO DE SITUACION FINANCIERA,2.6609,0.9066,5.2273,0.9116,5.2270,1.0651,2.6606,1.0600
2,1,3,C,SEPTIEMBRE 30 DE 2023,3.0685,1.0885,4.8062,1.0950,4.8057,1.2439,3.0679,1.2374
3,1,4,C,(Cifras en pesos),3.3562,1.2761,4.5234,1.2812,4.5227,1.4424,3.3555,1.4373
4,1,5,D,jun-23,4.4577,1.6108,4.8654,1.6057,4.8672,1.7485,4.4595,1.7536
5,1,5,G,sep-23,6.0879,1.6236,6.5157,1.6203,6.5168,1.7562,6.0889,1.7595
6,1,6,A,Código ACTIVO CORRIENTE,0.8195,1.7894,2.6497,1.7918,2.6495,1.9487,0.8193,1.9471
7,1,6,D,252.163.930.820,4.4850,1.8129,5.4706,1.8168,5.4701,1.9458,4.4844,1.9419
8,1,6,G,280.380.428.180,6.1188,1.8169,7.1037,1.8183,7.1035,1.9511,6.1186,1.9497
9,1,7,A,11 EQUIVALENTE AL EFECTIVO,1.1466,2.1688,3.1736,2.1735,3.1732,2.3226,1.1463,2.3178


----
#Exportación de la información formateada a archivo Excel

----

In [71]:
# Convertir el dataframe a un archivo excel con la estructura del PDF leído

# Función para exportar df, df_1 y hojas por página
def exportar_excel(df, df_1, excel_out):

    # Crear archivo Excel con las dos primeras hojas
    with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:

        # Hoja: Texto-Original
        df.to_excel(writer, sheet_name="Texto-Original", index=False)

        # Hoja: Texto-Ajustado
        df_1.to_excel(writer, sheet_name="Texto-Ajustado", index=False)

        # 2) Crear hojas por página
        paginas = df_1["pagina"].unique()

        for p in paginas:
            hoja_nombre = f"Texto-Pagina-{p}"

            # Agregar hoja en blanco
            ws = writer.book.create_sheet(hoja_nombre)

            # Filtrar textos de esa página
            df_pag = df_1[df_1["pagina"] == p]

            # Escribir cada texto en su (fila, columna)
            for _, row in df_pag.iterrows():

                fila = int(row["fila"])
                col_letra = row["columna"]

                celda = f"{col_letra}{fila}"
                texto = str(row["texto"]) if row["texto"] is not None else ""

                ws[celda] = texto

        # Guardar el archivo
        writer.book.save(excel_out)

exportar_excel(df, df_1, excel_out)

---
# **Interpretación y Mapeo del Contenido con Microsoft Azure OpenAI**

---

---
#Lectura y limpieza del documento extraído

---

In [72]:
# Leer y limpiar desde Excel (múltiples hojas)
# NOTA: SE INTENTÓ USAR OPENAI PARA LEER DIRECTAMENTE EL PDF PERO EL PROCESO REQUIERE PREVIAMENTE PROCESARLO CON AI-OCR

# Función para crear texto plano a partir del archivo Excel generado por el proceso de AI-OCR
def extract_text_from_excel(file_path):
    text = ""
    xls = pd.ExcelFile(file_path)

    for sheet_name in xls.sheet_names:
        # Excluir hojas que no se requieren para la interpretación
        if sheet_name in ["Texto-Original", "Texto-Ajustado"]:
            continue

        df = pd.read_excel(file_path, sheet_name=sheet_name, dtype=str)
        df = df.fillna("")

        for _, row in df.iterrows():
            row_text = " ".join([str(cell) for cell in row if str(cell).strip() != ""])
            if row_text:
                text += row_text + "\n"

    # Limpieza básica
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'Página \d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s{2,}', ' ', text)

    return text

# Ejecutar la función para crear texto plano a partir del archivo Excel generado por el proceso de AI-OCR
excel_text = extract_text_from_excel(file_input)

----
#Lectura de la taxonomía definida

----

In [73]:
# Leer la taxonomía resultado definida para el balance general
bg_taxonomy = pd.read_excel(file_taxonomy,
                           sheet_name="Taxonomia-Base-BG")

# Leer la taxonomía resultado definida para el estado de resultados
eerr_taxonomy = pd.read_excel(file_taxonomy,
                             sheet_name="Taxonomia-Base-EERR")

# Tomar los nombres de las cuentas de las taxonomías
bg_accounts = bg_taxonomy.iloc[:,0].dropna().tolist()
eerr_accounts = eerr_taxonomy.iloc[:,0].dropna().tolist()

----
# Definición y configuración del LLM Azure OpenAI para interpretar los estados financieros leídos con AI-OCR según la taxonomía

----

In [74]:
# Definir y configurar el LLM Azure OpenAI para interpretar los estados financieros leídos con AI-OCR según la taxonomía

# Asignar los parámetros requeridos por Azure OpenAI
client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)

# Función para usar Azure OpenAI (mapeo inteligente)
def map_accounts_with_llm(text, taxonomy, tipo_estado):
    prompt = f"""
Eres un experto contable colombiano.

Tienes un texto extraído de un estado financiero desordenado.
Debes mapear cada cuenta a una taxonomía estándar.

TIPO: {tipo_estado}

TAXONOMÍA:
{taxonomy}

REGLAS:
- Identifica cuentas y valores numéricos por cada periodo
- Trata de ajustar las cuentas de acuerdo con el nombre en la taxonomía que es de la forma "GRUPO - SUBGRUPO - CUENTA"
- Asume como válida la cuenta cuyo nombre esté completa o parcialmente en la taxonomía
- Asume como válida la cuenta cuyo nombre contenga completa o parcialmente el nombre en la taxonomía
- No tengas en cuenta las diferencias por mayúsculas y minúsculas
- Respeta el agrupamiento de las cuentas de acuerdo como se indica en la taxonomía
- El nombre de la cuenta que se genere en el resultado debe tener el nombre de la cuenta original y el nomnbre de la cuenta en la taxonomía
- Ignora encabezados, logos, ruido
- Ten en cuenta todas y cada una de las cuentas; Si no encuentras coincidencia, asigna:
  "SIN IDENTIFICAR (texto original con periodos y valores)"
- Mantén valores numéricos
- Responde SOLO en JSON así:

[
  {{
    "cuenta_original": "...",
    "cuenta_taxonomia": "...",
    "periodo": ...,
    "valor": ...
  }}
]

TEXTO:
{text[:12000]}
"""

    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": "Eres un analista financiero experto en NIIF Colombia"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    raw_content = response.choices[0].message.content
    start_idx = raw_content.find('[')
    end_idx = raw_content.rfind(']')

    if start_idx != -1 and end_idx != -1 and start_idx < end_idx:
        json_string = raw_content[start_idx : end_idx + 1]
    else:
        print(f"Alerta: los delimitadores del JSON '[' and ']' no se encontraron o se perdieron en la respuesta del LLM. Intentando un string simple. Content:\n{raw_content}")
        json_string = raw_content.strip('```json').strip('```').strip()

    return json_string.strip()

----
# Mapeo Inteligente para interpretar los estados financieros leídos con AI-OCR según la taxonomía usando LLM Azure OpenAI

----

In [75]:
# Ejecutar función para usar Azure OpenAI (mapeo inteligente)
bg_json = map_accounts_with_llm(excel_text, bg_accounts, "BALANCE GENERAL")
eerr_json = map_accounts_with_llm(excel_text, eerr_accounts, "ESTADO DE RESULTADO")

# Leer los JSON resultado del mapeo inteligente
bg_data = json.loads(bg_json)
eerr_data = json.loads(eerr_json)

In [76]:
# Mostrar resultado del mapeo inteligente del balance general
print (bg_json)
print(bg_data)

[
  {
    "cuenta_original": "Código ACTIVO CORRIENTE",
    "cuenta_taxonomia": "ACTIVOS - ACTIVOS CORRIENTES - SIN IDENTIFICAR",
    "periodo": "jun-23",
    "valor": 252163930820
  },
  {
    "cuenta_original": "Código ACTIVO CORRIENTE",
    "cuenta_taxonomia": "ACTIVOS - ACTIVOS CORRIENTES - SIN IDENTIFICAR",
    "periodo": "sep-23",
    "valor": 280380428180
  },
  {
    "cuenta_original": "11 EQUIVALENTE AL EFECTIVO",
    "cuenta_taxonomia": "ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA, BANCOS)",
    "periodo": "jun-23",
    "valor": 97793793418
  },
  {
    "cuenta_original": "11 EQUIVALENTE AL EFECTIVO",
    "cuenta_taxonomia": "ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA, BANCOS)",
    "periodo": "sep-23",
    "valor": 90591921820
  },
  {
    "cuenta_original": "1105 Caja",
    "cuenta_taxonomia": "ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA, BANCOS)",
    "periodo": "jun-23",
    "valor": 9500000
  },
  {
    "cuenta_original": "1105 Caja",
    "cuenta_taxonomia": "ACTI

In [77]:
# Mostrar resultado del mapeo inteligente del estado de resultado
print(eerr_json)
print(eerr_data)

[]
[]


In [78]:
# Normalizar resultados (incluye SIN IDENTIFICAR)

# Función para normalizar los resultados
def normalize_results(data):
    rows = []

    for item in data:
        cuenta = item.get("cuenta_taxonomia", "")
        valor = item.get("valor", 0)
        periodo = item.get("periodo", "")
        original = item.get("cuenta_original", "")

        if "SIN IDENTIFICAR" in cuenta:
            #cuenta = f"SIN IDENTIFICAR ({original})"
            cuenta = "SIN IDENTIFICAR"

        rows.append({
            "Cuenta": cuenta,
            "Original": original,
            "Periodo": periodo,
            "Valor": valor
        })

    return pd.DataFrame(rows)

# Ejecutar la función para normalizar los resultados del balance general y del estado de resultados
df_bg = normalize_results(bg_data)
df_eerr = normalize_results(eerr_data)

In [79]:
# Mostrar el dataset con la normalización del balance general
df_bg

,Cuenta,Original,Periodo,Valor
0,SIN IDENTIFICAR,Código ACTIVO CORRIENTE,jun-23,252163930820
1,SIN IDENTIFICAR,Código ACTIVO CORRIENTE,sep-23,280380428180
2,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",11 EQUIVALENTE AL EFECTIVO,jun-23,97793793418
3,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",11 EQUIVALENTE AL EFECTIVO,sep-23,90591921820
4,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",1105 Caja,jun-23,9500000
5,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",1105 Caja,sep-23,9500000
6,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",1110 Depositos e instituciones financieras,jun-23,97784293418
7,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",1110 Depositos e instituciones financieras,sep-23,90582421820
8,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",1132 Efectivo de uso restringido,jun-23,0
9,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",1132 Efectivo de uso restringido,sep-23,0


In [80]:
# Mostrar el dataset con la normalización del estado de resultados
df_eerr

""


----
# Generación del archivo excel final completamente interpretado y mapeado

----

In [81]:
# Generar Excel Final

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_bg.to_excel(writer, sheet_name="BG", index=False)
    df_eerr.to_excel(writer, sheet_name="EERR", index=False)

print("Archivo generado:", output_file)

Archivo generado: /content/drive/MyDrive/Colab Notebooks/PRUEBA_03_LLM204.xlsx
